# CyberFlow AI Engine — Model Training Notebook

**SIH 2026 — Ministry of Home Affairs / I4C Theme**

This notebook trains **real XGBoost classifiers** on synthetic fraud transaction data to power the CyberFlow AI engine.

### What gets trained:
1. **State Classifier** — Predicts operation state: `emerging`, `collection`, `distribution`, `layering`, `consolidation`, `cashout_prep`
2. **Next-Action Predictor** — Predicts: `cashout`, `further_layering`, `external_transfer`, `other` (with probability distribution)
3. **Risk Scorer** — Predicts network risk score (0.0–1.0)
4. **Priority Classifier** — Predicts intervention priority: `HIGH`, `MEDIUM`, `LOW`

### Data:
All training data is **100% synthetic**. No real NCRP/I4C/bank data is used.

---

## How to use this notebook:
1. Upload `training_features.csv` from `/ai-engine/data/` to Colab (or mount your Drive)
2. Run all cells
3. Download the 4 trained model `.json` files from the output
4. Place them in `/ai-engine/models/`
5. Run `python pipeline.py` — it will auto-detect and use the trained models

## 1. Setup & Install Dependencies

In [ ]:
!pip install -q xgboost scikit-learn pandas numpy matplotlib seaborn

import numpy as np
import pandas as pd
import xgboost as xgb
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
    mean_squared_error, mean_absolute_error, r2_score
)
from sklearn.preprocessing import LabelEncoder
import json
import os

print(f"XGBoost version: {xgb.__version__}")
print("Environment ready!")

## 2. Load Training Data

Upload `training_features.csv` to Colab first, or set the path below.

In [ ]:
# === CHANGE THIS PATH if needed ===
# For Colab: upload training_features.csv and use 'training_features.csv'
# For local: use the full path to ai-engine/data/training_features.csv
DATA_PATH = "training_features.csv"

# Try Colab upload if file not found
if not os.path.exists(DATA_PATH):
    try:
        from google.colab import files
        print("Please upload training_features.csv:")
        uploaded = files.upload()
        DATA_PATH = list(uploaded.keys())[0]
    except ImportError:
        raise FileNotFoundError(
            f"Cannot find {DATA_PATH}. "
            "Please upload training_features.csv or set the correct path."
        )

df = pd.read_csv(DATA_PATH)
print(f"Loaded {len(df)} cases with {len(df.columns)} columns")
print(f"\nColumn list:\n{list(df.columns)}")
df.head()

## 3. Explore the Data

In [ ]:
print("=== Data Shape ===")
print(f"Cases: {len(df)}")
print(f"Features: {len(df.columns) - 6}")
print()

print("=== Label Distributions ===")
for col in ['fraud_type', 'current_state', 'predicted_next_action', 'intervention_priority']:
    print(f"\n{col}:")
    print(df[col].value_counts().to_string())

print(f"\nnetwork_risk: mean={df['network_risk'].mean():.3f}, std={df['network_risk'].std():.3f}, range=[{df['network_risk'].min():.2f}, {df['network_risk'].max():.2f}]")
print(f"potential_exposure_inr: mean={df['potential_exposure_inr'].mean():.0f}, range=[{df['potential_exposure_inr'].min():.0f}, {df['potential_exposure_inr'].max():.0f}]")

In [ ]:
# Visualize distributions
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# State distribution by fraud type
pd.crosstab(df['fraud_type'], df['current_state']).plot(kind='bar', ax=axes[0,0])
axes[0,0].set_title('Operation State by Fraud Type')
axes[0,0].set_xlabel('')
axes[0,0].tick_params(axis='x', rotation=15)

# Risk distribution
for ft in df['fraud_type'].unique():
    axes[0,1].hist(df[df['fraud_type']==ft]['network_risk'], alpha=0.5, label=ft, bins=15)
axes[0,1].set_title('Network Risk Distribution')
axes[0,1].legend(fontsize=8)

# Next action distribution
pd.crosstab(df['fraud_type'], df['predicted_next_action']).plot(kind='bar', ax=axes[1,0])
axes[1,0].set_title('Next Action by Fraud Type')
axes[1,0].set_xlabel('')
axes[1,0].tick_params(axis='x', rotation=15)

# Priority distribution
df['intervention_priority'].value_counts().plot(kind='pie', ax=axes[1,1], autopct='%1.0f%%')
axes[1,1].set_title('Intervention Priority')

plt.tight_layout()
plt.show()

## 4. Prepare Feature Matrix & Labels

In [ ]:
# Feature columns — exactly matching generate_training_data.py FEATURE_COLS
FEATURE_COLS = [
    "total_volume", "tx_count", "max_amount", "min_amount", "mean_amount",
    "std_amount", "velocity_tx_per_min", "time_span_sec",
    "min_inter_arrival_sec", "mean_inter_arrival_sec", "std_inter_arrival_sec",
    "rapid_tx_count", "burst_ratio",
    "num_nodes", "num_edges", "max_fan_in", "max_fan_out",
    "mean_fan_in", "mean_fan_out", "sources_only_count", "sinks_only_count",
    "convergence_ratio", "max_betweenness", "mean_betweenness", "hop_depth",
    "zone_a_frac", "zone_b_frac", "zone_c_frac", "num_zones_active",
    "ft_investment_scam", "ft_digital_arrest", "ft_fake_payment_gateway",
]

X = df[FEATURE_COLS].copy()

# Handle any NaN/inf
X = X.replace([np.inf, -np.inf], np.nan).fillna(0)

print(f"Feature matrix shape: {X.shape}")
print(f"Any NaN remaining: {X.isna().any().any()}")

# Encode labels
STATE_CLASSES = ["emerging", "collection", "distribution", "layering", "consolidation", "cashout_prep"]
ACTION_CLASSES = ["cashout", "further_layering", "external_transfer", "other"]
PRIORITY_CLASSES = ["LOW", "MEDIUM", "HIGH"]

state_encoder = LabelEncoder()
state_encoder.classes_ = np.array(STATE_CLASSES)
y_state = state_encoder.transform(df['current_state'])

action_encoder = LabelEncoder()
action_encoder.classes_ = np.array(ACTION_CLASSES)
y_action = action_encoder.transform(df['predicted_next_action'])

priority_encoder = LabelEncoder()
priority_encoder.classes_ = np.array(PRIORITY_CLASSES)
y_priority = priority_encoder.transform(df['intervention_priority'])

y_risk = df['network_risk'].values

print(f"\nState classes:    {STATE_CLASSES}")
print(f"Action classes:   {ACTION_CLASSES}")
print(f"Priority classes: {PRIORITY_CLASSES}")
print(f"Risk range:       [{y_risk.min():.2f}, {y_risk.max():.2f}]")

In [ ]:
# Train/test split (80/20)
X_train, X_test, y_state_train, y_state_test = train_test_split(
    X, y_state, test_size=0.2, random_state=42, stratify=y_state
)
_, _, y_action_train, y_action_test = train_test_split(
    X, y_action, test_size=0.2, random_state=42, stratify=y_action
)
_, _, y_risk_train, y_risk_test = train_test_split(
    X, y_risk, test_size=0.2, random_state=42
)
_, _, y_priority_train, y_priority_test = train_test_split(
    X, y_priority, test_size=0.2, random_state=42, stratify=y_priority
)

print(f"Train: {len(X_train)} cases | Test: {len(X_test)} cases")

## 5. Train XGBoost Models

### 5a. Operation State Classifier

In [ ]:
state_model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    objective='multi:softprob',
    num_class=len(STATE_CLASSES),
    eval_metric='mlogloss',
    random_state=42,
    use_label_encoder=False
)

state_model.fit(X_train, y_state_train, eval_set=[(X_test, y_state_test)], verbose=False)

y_state_pred = state_model.predict(X_test)
print("=== Operation State Classifier ===")
print(f"Accuracy: {accuracy_score(y_state_test, y_state_pred):.3f}")
print()
print(classification_report(y_state_test, y_state_pred, target_names=STATE_CLASSES, zero_division=0))

In [ ]:
# Confusion matrix for state classifier
cm = confusion_matrix(y_state_test, y_state_pred)
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=STATE_CLASSES, yticklabels=STATE_CLASSES, ax=ax)
ax.set_title('State Classification — Confusion Matrix')
ax.set_ylabel('True State')
ax.set_xlabel('Predicted State')
plt.tight_layout()
plt.show()

### 5b. Next-Action Predictor

In [ ]:
action_model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    objective='multi:softprob',
    num_class=len(ACTION_CLASSES),
    eval_metric='mlogloss',
    random_state=42,
    use_label_encoder=False
)

action_model.fit(X_train, y_action_train, eval_set=[(X_test, y_action_test)], verbose=False)

y_action_pred = action_model.predict(X_test)
print("=== Next-Action Predictor ===")
print(f"Accuracy: {accuracy_score(y_action_test, y_action_pred):.3f}")
print()
print(classification_report(y_action_test, y_action_pred, target_names=ACTION_CLASSES, zero_division=0))

### 5c. Network Risk Scorer (Regression)

In [ ]:
risk_model = xgb.XGBRegressor(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.1,
    objective='reg:squarederror',
    random_state=42
)

risk_model.fit(X_train, y_risk_train, eval_set=[(X_test, y_risk_test)], verbose=False)

y_risk_pred = risk_model.predict(X_test)
y_risk_pred = np.clip(y_risk_pred, 0.0, 1.0)  # Ensure valid range

print("=== Network Risk Scorer ===")
print(f"MAE:  {mean_absolute_error(y_risk_test, y_risk_pred):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_risk_test, y_risk_pred)):.4f}")
print(f"R2:   {r2_score(y_risk_test, y_risk_pred):.4f}")

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y_risk_test, y_risk_pred, alpha=0.5, s=20)
ax.plot([0, 1], [0, 1], 'r--', label='Perfect')
ax.set_xlabel('True Risk')
ax.set_ylabel('Predicted Risk')
ax.set_title('Risk Scorer — True vs Predicted')
ax.legend()
plt.tight_layout()
plt.show()

### 5d. Intervention Priority Classifier

In [ ]:
priority_model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.1,
    objective='multi:softprob',
    num_class=len(PRIORITY_CLASSES),
    eval_metric='mlogloss',
    random_state=42,
    use_label_encoder=False
)

priority_model.fit(X_train, y_priority_train, eval_set=[(X_test, y_priority_test)], verbose=False)

y_priority_pred = priority_model.predict(X_test)
print("=== Intervention Priority Classifier ===")
print(f"Accuracy: {accuracy_score(y_priority_test, y_priority_pred):.3f}")
print()
print(classification_report(y_priority_test, y_priority_pred, target_names=PRIORITY_CLASSES, zero_division=0))

## 6. Feature Importance Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(18, 14))

models_info = [
    (state_model, 'State Classifier'),
    (action_model, 'Next-Action Predictor'),
    (risk_model, 'Risk Scorer'),
    (priority_model, 'Priority Classifier')
]

for idx, (model, title) in enumerate(models_info):
    ax = axes[idx // 2][idx % 2]
    importances = model.feature_importances_
    top_k = 15
    top_indices = np.argsort(importances)[-top_k:]
    ax.barh(
        [FEATURE_COLS[i] for i in top_indices],
        importances[top_indices]
    )
    ax.set_title(f'{title} — Top {top_k} Features')

plt.tight_layout()
plt.show()

## 7. Cross-Validation (Optional — Full Dataset)

In [ ]:
print("=== 5-Fold Cross-Validation ===")

cv_state = cross_val_score(state_model, X, y_state, cv=5, scoring='accuracy')
print(f"State Classifier:    {cv_state.mean():.3f} (+/- {cv_state.std():.3f})")

cv_action = cross_val_score(action_model, X, y_action, cv=5, scoring='accuracy')
print(f"Action Predictor:    {cv_action.mean():.3f} (+/- {cv_action.std():.3f})")

cv_risk = cross_val_score(risk_model, X, y_risk, cv=5, scoring='r2')
print(f"Risk Scorer (R2):    {cv_risk.mean():.3f} (+/- {cv_risk.std():.3f})")

cv_priority = cross_val_score(priority_model, X, y_priority, cv=5, scoring='accuracy')
print(f"Priority Classifier: {cv_priority.mean():.3f} (+/- {cv_priority.std():.3f})")

## 8. Save Trained Models

Saves as XGBoost native JSON format (portable, no pickle needed).

In [ ]:
MODEL_DIR = "models"
os.makedirs(MODEL_DIR, exist_ok=True)

state_model.save_model(os.path.join(MODEL_DIR, "state_classifier.json"))
action_model.save_model(os.path.join(MODEL_DIR, "action_predictor.json"))
risk_model.save_model(os.path.join(MODEL_DIR, "risk_scorer.json"))
priority_model.save_model(os.path.join(MODEL_DIR, "priority_classifier.json"))

# Save class mappings alongside models
metadata = {
    "state_classes": STATE_CLASSES,
    "action_classes": ACTION_CLASSES,
    "priority_classes": PRIORITY_CLASSES,
    "feature_columns": FEATURE_COLS,
    "training_samples": len(df),
    "test_accuracy": {
        "state": float(accuracy_score(y_state_test, y_state_pred)),
        "action": float(accuracy_score(y_action_test, y_action_pred)),
        "priority": float(accuracy_score(y_priority_test, y_priority_pred)),
        "risk_r2": float(r2_score(y_risk_test, y_risk_pred)),
        "risk_mae": float(mean_absolute_error(y_risk_test, y_risk_pred))
    }
}
with open(os.path.join(MODEL_DIR, "model_metadata.json"), "w") as f:
    json.dump(metadata, f, indent=2)

print("=== Models Saved ===")
for f_name in os.listdir(MODEL_DIR):
    f_path = os.path.join(MODEL_DIR, f_name)
    size_kb = os.path.getsize(f_path) / 1024
    print(f"  {f_name}: {size_kb:.1f} KB")

## 9. Download Models (Colab Only)

Run this cell to download the model files. Place them in `/ai-engine/models/`.

In [ ]:
try:
    from google.colab import files
    for f_name in os.listdir(MODEL_DIR):
        files.download(os.path.join(MODEL_DIR, f_name))
    print("Downloaded all model files!")
except ImportError:
    print("Not running in Colab. Models saved locally in:", os.path.abspath(MODEL_DIR))

## 10. Test Prediction on a New Case

Quick sanity check: predict on a case and see the output.

In [ ]:
# Pick one test sample
sample_idx = 0
sample_X = X_test.iloc[[sample_idx]]

# State prediction
state_pred_idx = state_model.predict(sample_X)[0]
state_proba = state_model.predict_proba(sample_X)[0]
pred_state = STATE_CLASSES[state_pred_idx]

# Action prediction
action_pred_idx = action_model.predict(sample_X)[0]
action_proba = action_model.predict_proba(sample_X)[0]
pred_action = ACTION_CLASSES[action_pred_idx]

# Risk prediction
pred_risk = float(np.clip(risk_model.predict(sample_X)[0], 0.0, 1.0))

# Priority prediction
priority_pred_idx = priority_model.predict(sample_X)[0]
pred_priority = PRIORITY_CLASSES[priority_pred_idx]

print("=== Sample Prediction ===")
print(f"Predicted State:    {pred_state}")
print(f"  State probs:      {dict(zip(STATE_CLASSES, [round(p,3) for p in state_proba]))}")
print(f"Predicted Action:   {pred_action}")
print(f"  Action probs:     {dict(zip(ACTION_CLASSES, [round(p,3) for p in action_proba]))}")
print(f"Predicted Risk:     {pred_risk:.3f}")
print(f"Predicted Priority: {pred_priority}")
print()
print(f"True State:    {STATE_CLASSES[y_state_test[sample_idx]]}")
print(f"True Action:   {ACTION_CLASSES[y_action_test[sample_idx]]}")
print(f"True Risk:     {y_risk_test[sample_idx]:.3f}")
print(f"True Priority: {PRIORITY_CLASSES[y_priority_test[sample_idx]]}")

---

## Summary

You now have 4 trained XGBoost models:

| Model | File | Task | Output |
|-------|------|------|--------|
| State Classifier | `state_classifier.json` | 6-class classification | `emerging`, `collection`, `distribution`, `layering`, `consolidation`, `cashout_prep` |
| Action Predictor | `action_predictor.json` | 4-class classification w/ probabilities | `cashout`, `further_layering`, `external_transfer`, `other` |
| Risk Scorer | `risk_scorer.json` | Regression [0, 1] | Continuous risk score |
| Priority Classifier | `priority_classifier.json` | 3-class classification | `HIGH`, `MEDIUM`, `LOW` |

### Next steps:
1. Place the `.json` model files in `/ai-engine/models/`
2. Run `python pipeline.py` — it will detect and use these real trained models instead of hardcoded rules
3. CF-1042 demo case values are calibrated via the pipeline to match the shared contract exactly

### Disclaimers
- All training data is **100% synthetic** — no real NCRP/I4C/bank data
- This is a **decision-support** tool, not an autonomous decision maker
- All outputs are **probabilistic**, not deterministic guarantees